# Library

In [10]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp

# Get Hollistic Point and Extract

In [11]:
mp_holistic = mp.solutions.holistic # Holistic model
mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [12]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR-CONVERSION BGR-to-RGB
    image.flags.writeable = False                  # Convert image to not-writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Convert image to writeable 
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR-COVERSION RGB-to-BGR
    return image, results

def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # Draw pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # Draw right hand connections

def draw_styled_landmarks(image, results):
    # Draw pose connections
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)) 
    # Draw left hand connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)) 
    # Draw right hand connections  
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2))
    
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*4)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, lh, rh])

In [13]:
def extract_keypoints(results):
    # ============================================
    # POSE: 33 landmark × (x,y,z,visibility)
    # Kita ambil hanya (x,y,z)
    # ============================================
    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z] for lm in results.pose_landmarks.landmark])
    else:
        pose = np.zeros((33, 3))

    # ============================================
    # LEFT HAND: 21 landmark × (x,y,z)
    # ============================================
    if results.left_hand_landmarks:
        lh = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark])
    else:
        lh = np.zeros((21, 3))

    # ============================================
    # RIGHT HAND: 21 landmark × (x,y,z)
    # ============================================
    if results.right_hand_landmarks:
        rh = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark])
    else:
        rh = np.zeros((21, 3))

    # Gabungkan → (75, 3)
    return np.vstack([pose, lh, rh])


In [14]:
def prob_viz(res, actions, input_frame, colors, top_k=3):
    output_frame = input_frame.copy()

    # Ambil indeks TOP-K probability (urut dari terbesar)
    top_indices = np.argsort(res)[-top_k:][::-1]

    for i, idx in enumerate(top_indices):
        prob = res[idx]
        label = actions[idx]

        cv2.rectangle(
            output_frame,
            (0, 60 + i * 40),
            (int(prob * 100), 90 + i * 40),
            colors[idx],
            -1
        )

        cv2.putText(
            output_frame,
            f"{label}: {prob:.2f}",
            (0, 85 + i * 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2,
            cv2.LINE_AA
        )

    return output_frame

# Classification Test

In [15]:
import tensorflow as tf
model = tf.keras.models.load_model('my_model.keras') # Model

# Actions that we try to detect
actions = np.array(['yang', 'dan', 'dengan', 'ini', 'untuk', 'dari', 'dalam', 'itu', 'tidak'])

colors = [(245,117,16) for _ in range(len(actions))]

# 1. New detection variables
sequence = []
sentence = []
predictions = []
threshold = 0.5

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(model_complexity=1,
    smooth_landmarks=True,
    min_detection_confidence=0.3,
    min_tracking_confidence=0.5) as holistic:

    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        
        # Draw landmarks
        draw_styled_landmarks(image, results)
        
        # 2. Prediction logic
        keypoints = extract_keypoints(results)
        sequence.append(keypoints)
        sequence = sequence[-30:]
        
        if len(sequence) == 30:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(actions[np.argmax(res)])
            predictions.append(np.argmax(res))
            
            
        #3. Viz logic
            if np.unique(predictions[-10:])[0]==np.argmax(res): 
                if res[np.argmax(res)] > threshold: 
                    
                    if len(sentence) > 0: 
                        if actions[np.argmax(res)] != sentence[-1]:
                            sentence.append(actions[np.argmax(res)])
                    else:
                        sentence.append(actions[np.argmax(res)])

            if len(sentence) > 5: 
                sentence = sentence[-5:]

            # Viz probabilities
            image = prob_viz(res, actions, image, colors)
            
        cv2.rectangle(image, (0,0), (640, 40), (245, 117, 16), -1)
        cv2.putText(image, ' '.join(sentence), (3,30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
        
        # Show to screen
        cv2.imshow('OpenCV Feed', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
yang
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
ini
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
ini
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
ini
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
ini
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
ini
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
ini
1/1 ━━━━━━━━━━━━━━━